# Assignment 02 — Application 2: House-Price Prediction

**Student:** &lt;name&gt; &lt;id&gt;  |  **Class:** &lt;class&gt;  |  **Date:** 2026-09-03  
**Lecturer:** Dinh Que Tran, Ph.D., Assoc. Prof.  |  **Semester:** I.2026

The 23 sections below take the raw Vietnamese real-estate listings CSV through

```mermaid
flowchart LR
    Data --> Understand --> Clean --> Represent --> Learn --> Evaluate --> Persist --> Deploy
```

Every table or figure is followed by a short markdown interpretation. `RANDOM_SEED = 42`
everywhere. The persisted pipeline written in section 22 is what `../api/` serves.

## 0. Header & setup

Fix and print the environment (seed + library versions) so the experiment is reproducible.

In [ ]:
import sys, platform, random, time, warnings
import numpy as np, pandas as pd
import sklearn, matplotlib, matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

print('Python      :', sys.version.split()[0], 'on', platform.system(), platform.release())
print('numpy       :', np.__version__)
print('pandas      :', pd.__version__)
print('scikit-learn:', sklearn.__version__)
print('matplotlib  :', matplotlib.__version__)
try:
    import xgboost; print('xgboost     :', xgboost.__version__)
except ImportError:
    print('xgboost     : NOT INSTALLED  (pip install xgboost)')
print('RANDOM_SEED  =', RANDOM_SEED)

## 1. Problem definition

**Real-world problem.** Property portals and buyers need a fair price estimate for a
residential listing from its attributes, so an asking price can be sanity-checked before
it is published or accepted.

**Supervised task.** **Regression** — the target is a continuous amount, so the model
minimises an error loss (MSE) and is scored with MAE / RMSE / R², **not** accuracy or a
confusion matrix. This is the key difference from the diabetes classification app.

- `X` = property attributes: area, frontage/length, room counts, floors, alley width,
  property type, position, compass direction, road type, province, and a district
  parsed from the free-text location.
- `y` = `Price`, the listing price in **million VND** (title "2.7 tỷ" ↔ `Price` = 2700).

$$ X \in \mathbb{R}^{N \times d}, \qquad y \in \mathbb{R}^{N} $$

## 2. Dataset source

| Field | Value |
|---|---|
| **Name** | VN Real Estate Listings (April–September 2025) |
| **File** | `../data/VN-real-estate-Apr-Sept-2025.csv` |
| **Rows / cols (raw)** | 236,226 × 28 |
| **Kaggle URL** | *[FILL IN: exact Kaggle dataset URL]* |
| **Licence / version** | *[FILL IN: e.g. CC0 / scraped-data]*, downloaded 2025-09 |
| **Collection** | Scraped from a Vietnamese property-listing portal; listings updated Apr–Sept 2025 |
| **Encoding / sep** | UTF-8 **with BOM** (`utf-8-sig`), comma-separated |

Full column reference and known quality issues: `../data/README.md`.

## 3. Dataset loading

In [ ]:
CSV_PATH = '../data/VN-real-estate-Apr-Sept-2025.csv'   # separator ',', encoding utf-8-sig
df = pd.read_csv(CSV_PATH, encoding='utf-8-sig', low_memory=False)
df.head()

**Interpretation.** The file loads without error. One row is one **property listing**:
its attributes plus the asking `Price`. The `Title` and `Description` columns are free
text; `Title` embeds the price and area verbatim (e.g. "Bán nhà 78.7m² 2.7 tỷ …"), so it
**must be dropped** later — using it would leak the target.

## 4. Dataset inspection

In [ ]:
print('shape (rows, columns):', df.shape)     # <-- the dataset shape
df.info()
display(df.describe(include='number').T)
print('\nmissing per column:')
print(df.isna().sum().sort_values(ascending=False))
print('\nexact duplicate rows:', df.duplicated().sum())

**Interpretation.** `df.shape = (236226, 28)` — **236,226 listings**, **28 attributes**
(1 target `Price` + 27 candidate inputs). `describe()` exposes corruption in the two most
important numeric columns: `Price` ranges from `0` to `9.2e12` and `Area` from `-6` to
`1e18` — physically impossible, so these are data-entry errors handled in section 8.
Several columns are mostly missing (`Bathrooms` ~84%, `Floors` ~79%, `Bedrooms` ~72%,
`Direction` ~70%, `Road Type` ~67%). `VIP Account` is constant. 0 exact duplicate rows,
but `Listing ID` repeats — checked in section 7.

## 5. Data-quality analysis

In [ ]:
PRICE_LO, PRICE_HI = 0, 200_000        # million VND  (~0 .. ~200 tỷ)
AREA_LO, AREA_HI   = 0, 10_000         # m^2

rows = []
rows.append(['Price', 'value <= 0 (impossible)', int((df['Price'] <= PRICE_LO).sum()), 'drop row'])
rows.append(['Price', f'value > {PRICE_HI} (implausible outlier)', int((df['Price'] > PRICE_HI).sum()), 'drop row'])
rows.append(['Area',  'value <= 0 (impossible)', int((df['Area'] <= AREA_LO).sum()), 'drop row'])
rows.append(['Area',  f'value > {AREA_HI} (implausible outlier)', int((df['Area'] > AREA_HI).sum()), 'drop row'])
rows.append(['Area',  'missing', int(df['Area'].isna().sum()), 'drop row (target-critical size)'])
for c in ['Bedrooms', 'Bathrooms', 'Floors']:
    rows.append([c, 'missing', int(df[c].isna().sum()), 'median impute + missing-indicator flag'])
for c in ['Width', 'Length', 'Alley Width']:
    rows.append([c, 'missing', int(df[c].isna().sum()), 'median impute (in pipeline)'])
for c in ['Position', 'Direction', 'Road Type']:
    rows.append([c, 'missing', int(df[c].isna().sum()), "fill 'Unknown' category"])
rows.append(['VIP Account', 'constant column', int(df['VIP Account'].nunique()), 'drop column'])
rows.append(['Listing ID', 'duplicate ids', int(df['Listing ID'].duplicated().sum()), 'drop dupes by id (section 7)'])
rows.append(['Title', 'free text — embeds Price', df['Title'].notna().sum(), 'DROP (target leakage)'])
rows.append(['Description', 'free text', int(df['Description'].isna().sum()), 'drop column (not used for tabular model)'])
quality = pd.DataFrame(rows, columns=['column', 'issue', 'count', 'planned action'])
quality

**Explain the results.** The dominant problems are (a) **corrupted `Price` / `Area`**
— a small fraction of rows carry impossible or absurd values, dropped outright; (b)
**heavy missingness** in room-count and road columns — imputed, with a flag where the
column is >50% missing so the model can use "was this recorded?"; (c) **`VIP Account`
constant** and two **free-text columns** — removed. Actions are applied in sections 6–9
and 13–15.

## 6. Missing-value analysis

In [ ]:
miss = df.isna().sum()
miss_pct = (miss / len(df) * 100).round(1)
miss_tbl = pd.DataFrame({'missing_count': miss, 'missing_pct': miss_pct})
miss_tbl = miss_tbl[miss_tbl['missing_count'] > 0].sort_values('missing_pct', ascending=False)
miss_tbl

**Strategy per column (justified).**

| Column | ~% missing | Decision | Why |
|---|---|---|---|
| `Area` | ~0.1% | **drop the row** | size is the strongest price driver; too central to impute |
| `Width` | ~26% | median impute (pipeline) | roughly symmetric; correlated with `Area` |
| `Length` | ~58% | median impute (pipeline) | high, but adds shape signal; keep column |
| `Bedrooms` | ~72% | median impute **+ `Bedrooms_missing` flag** | often absent for land listings — missingness is informative |
| `Bathrooms` | ~84% | median impute **+ `Bathrooms_missing` flag** | same |
| `Floors` | ~79% | median impute **+ `Floors_missing` flag** | same |
| `Alley Width` | ~70% | median impute (pipeline) | only meaningful for alley properties |
| `Position` / `Direction` / `Road Type` | 40–70% | fill **'Unknown'** category | absence is a category, not a number |
| `Latitude` / `Longitude` | ~68% | **drop columns** | too sparse to rely on; province + district cover location |

All imputers are **fitted inside the pipeline on the training split only** (section 15).

## 7. Duplicate analysis

In [ ]:
n_before = len(df)
print('exact duplicate rows        :', df.duplicated().sum())
print('duplicate Listing ID values :', df['Listing ID'].duplicated().sum())

# Listing ID is a natural key — the same advert re-scraped on different days.
df = df.drop_duplicates(subset='Listing ID', keep='last').reset_index(drop=True)
print(f'rows: {n_before} -> {len(df)}  (removed {n_before - len(df)} re-scrapes)')

**Interpretation.** There are no byte-identical rows, but ~19.5k listings share a
`Listing ID` — the same advert captured on more than one scrape date. Keeping all of
them would (i) inflate `N`, (ii) let the *same* property fall in both train and test,
leaking information. We keep the **last** capture per `Listing ID`. `N` drops accordingly.

## 8. Invalid-value analysis

In [ ]:
checks = {
    'Price <= 0'            : (df['Price'] <= PRICE_LO).sum(),
    f'Price > {PRICE_HI}'   : (df['Price'] > PRICE_HI).sum(),
    'Area <= 0'             : (df['Area'] <= AREA_LO).sum(),
    f'Area > {AREA_HI}'     : (df['Area'] > AREA_HI).sum(),
    'Bedrooms < 0'          : (df['Bedrooms'] < 0).sum(),
    'Floors < 0'            : (df['Floors'] < 0).sum(),
}
print(pd.Series(checks, name='violations'))

n_before = len(df)
df = df[(df['Price'] > PRICE_LO) & (df['Price'] <= PRICE_HI)]
df = df[(df['Area'].notna()) & (df['Area'] > AREA_LO) & (df['Area'] <= AREA_HI)]
df = df.reset_index(drop=True)
print(f'\nrows after domain filter: {n_before} -> {len(df)}  ({len(df)/n_before*100:.1f}% kept)')

**Treatment and why.** A price of 0 (or 9e12 VND) and an area of −6 (or 1e18 m²) cannot
describe a real property — they are recording errors, not signal, and would dominate any
error-based loss. We **drop** those rows rather than cap them, because the corrupt values
give no usable information about price. The bounds (`0 < Price ≤ 200,000` million VND,
`0 < Area ≤ 10,000` m²) keep ~93% of listings. *(Diabetes analogue: `Glucose == 0`.)*

## 9. Outlier analysis

In [ ]:
num_cols_raw = ['Price', 'Area', 'Width', 'Length', 'Bedrooms', 'Bathrooms', 'Floors', 'Alley Width']
Q1, Q3 = df[num_cols_raw].quantile(0.25), df[num_cols_raw].quantile(0.75)
IQR = Q3 - Q1
iqr_outliers = ((df[num_cols_raw] < Q1 - 1.5*IQR) | (df[num_cols_raw] > Q3 + 1.5*IQR)).sum()
print('IQR-rule outlier counts:')
print(iqr_outliers)

fig, ax = plt.subplots(2, 4, figsize=(15, 6))
for a, c in zip(ax.ravel(), num_cols_raw):
    df.boxplot(column=c, ax=a); a.set_title(c)
plt.tight_layout(); plt.show()

**Decision.** `Price`, `Area`, `Width` and `Length` have long right tails, but a 15 tỷ
villa on 800 m² is a **genuine** high-end property, not an error (the impossible values
were already removed in section 8). Deleting these rows would bias the model low and
shrink coverage of the exact segment users care about. We therefore **keep** them and
instead **model `log1p(Price)`** (section 12), which compresses the tail so linear /
distance-based models are not dominated by a few large listings. Extreme `Area` / `Width`
are additionally **capped at the 99th percentile** inside feature engineering (section 13).

## 10. Exploratory data analysis

Organised in three subsections, **one figure (or one closely-related pair) per block**,
each block followed by an *Observation / Interpretation / ML implication* note:

- **10.1 Distributions** — the target and the individual features.
- **10.2 Relationships & correlation** — each feature against the target, then the full
  correlation analysis (numeric Pearson *and* categorical correlation-ratio η²).
- **10.3 Supporting analyses** — missingness, skew, and the per-type area effect.

All plots use the cleaned frame from sections 7–8; scatter plots are on a random sample.

### 10.1 Distributions

The target on three scales (10.1.1–10.1.2) and the most important individual features
(10.1.3–10.1.5).

In [ ]:
# derived columns used throughout section 10
df['price_log'] = np.log1p(df['Price'])
df['price_per_m2'] = df['Price'] / df['Area']
df['area_log'] = np.log1p(df['Area'])

In [ ]:
# ---- 10.1.1 · target distribution: raw Price vs log1p(Price) ----
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(df['Price'], bins=80)
ax[0].set_title('10.1.1a · Price (million VND) — raw')
ax[0].set_xlabel('Price'); ax[0].set_ylabel('listings')
ax[1].hist(df['price_log'], bins=80)
ax[1].set_title('10.1.1b · log1p(Price) — the modelling target')
ax[1].set_xlabel('log1p(Price)')
plt.tight_layout(); plt.show()

print('Price skew  :', round(df['Price'].skew(), 2))
print('log1p skew  :', round(df['price_log'].skew(), 2))

**Observation.** Raw `Price` is extreme right-skew — most listings sit at 2–5 tỷ with a
thin tail to ~200 tỷ (skew ≈ 3.4). `log1p(Price)` is close to symmetric (skew ≈ 0).
**Interpretation.** Price behaves multiplicatively, not additively.
**ML implication.** The model is trained on `log1p(Price)` and the metrics are reported
back on the million-VND scale with `expm1` (sections 12, 16, 19).

In [ ]:
# ---- 10.1.2 · unit price (Price per m²) ----
pm2 = df['price_per_m2'].clip(upper=df['price_per_m2'].quantile(0.99))
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(pm2, bins=80)
ax.set_title('10.1.2 · Price per m² (clipped at the 99th percentile)')
ax.set_xlabel('million VND / m²'); ax.set_ylabel('listings')
plt.tight_layout(); plt.show()

print(df['price_per_m2'].describe(percentiles=[.5, .9, .99]).round(1).to_string())

**Observation.** Unit price is also right-skewed; the median is a few tens of
million VND/m² with a long upper tail (central-district listings).
**Interpretation.** `Price / Area` is not a stable constant — it varies strongly with
location and property type, so it cannot replace the model.
**ML implication.** `price_per_m2` is used only for EDA and for the API response; it is
**not** a training feature (it is a function of the target).

In [ ]:
# ---- 10.1.3 · log1p(Area) ----
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df['area_log'], bins=80)
ax.set_title('10.1.3 · log1p(Area)')
ax.set_xlabel('log1p(Area m²)'); ax.set_ylabel('listings')
plt.tight_layout(); plt.show()

print('Area skew        :', round(df['Area'].skew(), 2))
print('log1p(Area) skew :', round(df['area_log'].skew(), 2))

**Observation.** Raw `Area` is heavily right-skewed (skew ≈ 5); on the `log1p` scale it
is roughly bell-shaped (skew ≈ 1.3) after the section-8 domain filter.
**Interpretation.** A few very large plots would otherwise dominate distance- and
error-based models.
**ML implication.** `Area` (and `Width` / `Length` / `Alley Width`) are capped at the
99th percentile in section 13, and every numeric column is standardised in section 15.

In [ ]:
# ---- 10.1.4 · number of listings per property type ----
fig, ax = plt.subplots(figsize=(9, 4))
df['Property Type'].value_counts().plot(kind='barh', ax=ax)
ax.invert_yaxis()
ax.set_title('10.1.4 · listings per Property Type'); ax.set_xlabel('count')
plt.tight_layout(); plt.show()

print(df['Property Type'].value_counts().to_string())

**Observation.** `Đất` (land) and `Nhà riêng` (houses) make up the large majority of
listings; `Khách sạn`, `Văn phòng`, `Nhà trọ` have only a few hundred rows each.
**Interpretation.** The class mix is very unbalanced, and the rare types are exactly the
ones the model will struggle on (section 20).
**ML implication.** `Property Type` is one-hot encoded; the rare levels stay as their own
columns (7 values, well above the `min_frequency` threshold).

In [ ]:
# ---- 10.1.5 · room / floor counts (0–10, where recorded) ----
room = df[['Bedrooms', 'Bathrooms', 'Floors']].melt(var_name='feature', value_name='n')
room = room[room['n'].between(0, 10)]
fig, ax = plt.subplots(figsize=(8, 4))
room.boxplot(column='n', by='feature', ax=ax)
ax.set_title('10.1.5 · Bedrooms / Bathrooms / Floors (0–10)')
ax.set_xlabel(''); ax.set_ylabel('count'); plt.suptitle('')
plt.tight_layout(); plt.show()

print(df[['Bedrooms', 'Bathrooms', 'Floors']].describe().round(1).to_string())

**Observation.** When present, `Bedrooms` / `Bathrooms` / `Floors` cluster tightly at
2–4; but they are recorded for only ~30% of rows (mostly built property, not land).
**Interpretation.** These counts are meaningful for houses / apartments and largely
absent for land — the missingness itself is informative.
**ML implication.** Section 13 median-imputes the value **and** adds a
`<column>_missing` 0/1 indicator; see 10.3.2 for the by-type breakdown.

### 10.2 Relationships & correlation

Each feature against the target (10.2.1–10.2.3), then the full correlation analysis:
numeric Pearson (10.2.4) and categorical correlation-ratio η² (10.2.5).

In [ ]:
# correlation ratio (eta^2): share of variance in `values` explained by a categorical grouping
def correlation_ratio(categories, values):
    cats = categories.astype('category').cat.codes.values
    y = values.values
    ss_total = ((y - y.mean()) ** 2).sum()
    ss_between = sum(len(y[cats == k]) * (y[cats == k].mean() - y.mean()) ** 2
                     for k in np.unique(cats))
    return ss_between / ss_total if ss_total > 0 else 0.0

In [ ]:
# ---- 10.2.1 · log Price vs log Area, all property types pooled ----
samp = df.sample(min(8000, len(df)), random_state=RANDOM_SEED)
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(samp['area_log'], samp['price_log'], s=6, alpha=0.3)
ax.set_title('10.2.1 · log Price vs log Area (pooled)')
ax.set_xlabel('log1p(Area)'); ax.set_ylabel('log1p(Price)')
plt.tight_layout(); plt.show()

print('pooled Pearson r(log Area, log Price) =',
      round(df['area_log'].corr(df['price_log']), 3))

**Observation.** The pooled cloud shows only a **weak** positive trend — Pearson r ≈ 0.02
— with very wide vertical spread at every area.
**Interpretation.** Taken over all listings at once, area barely predicts price. This is
misleading (see 10.2.3 / 10.3.4): the land-vs-built mix cancels the effect out.
**ML implication.** Area cannot be used as a standalone linear predictor; it needs the
property-type context.

In [ ]:
# ---- 10.2.2 · price level by property type and by province ----
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
df.groupby('Property Type')['Price'].median().sort_values().plot(kind='barh', ax=ax[0])
ax[0].set_title('10.2.2a · median Price by Property Type'); ax[0].set_xlabel('median Price (M VND)')

top_prov = df['Province'].value_counts().head(15).index
(df[df['Province'].isin(top_prov)]
   .groupby('Province')['price_per_m2'].median().sort_values()
   .plot(kind='barh', ax=ax[1]))
ax[1].set_title('10.2.2b · median Price/m² — top-15 provinces'); ax[1].set_xlabel('M VND / m²')
plt.tight_layout(); plt.show()

**Observation.** Median price differs several-fold across property types, and median
Price/m² varies ~5× across provinces (HCMC / Hà Nội / Đà Nẵng highest, rural provinces
lowest).
**Interpretation.** Property type and province each set the *level* of price; area then
scales it within that level.
**ML implication.** Both are one-hot encoded (section 15); they are expected to be the
strongest features (confirmed by 10.2.5 and section 20's feature importances).

In [ ]:
# ---- 10.2.3 · the same relationship, split by property type ----
fig, ax = plt.subplots(1, 2, figsize=(16, 4.5))
for pt in df['Property Type'].value_counts().head(4).index:
    d = df[df['Property Type'] == pt].sample(
        min(2000, (df['Property Type'] == pt).sum()), random_state=RANDOM_SEED)
    ax[0].scatter(d['area_log'], d['price_log'], s=6, alpha=0.25, label=pt)
ax[0].legend(fontsize=8)
ax[0].set_title('10.2.3a · log Price vs log Area, by Property Type')
ax[0].set_xlabel('log1p(Area)'); ax[0].set_ylabel('log1p(Price)')

within = (df.groupby('Property Type')
            .apply(lambda g: g['area_log'].corr(g['price_log']))
            .sort_values())
within.plot(kind='barh', ax=ax[1], color='#55A868')
for i, v in enumerate(within.values):
    ax[1].text(v + (0.01 if v >= 0 else -0.01), i, f'{v:.2f}', va='center',
               ha='left' if v >= 0 else 'right', fontsize=9)
ax[1].axvline(0, color='k', lw=0.8)
ax[1].set_title('10.2.3b · corr(log Area, log Price) WITHIN each type')
ax[1].set_xlabel('Pearson r')
plt.tight_layout(); plt.show()

**Observation.** Split by type, the Area→Price relationship is clearly positive for the
built types (`Nhà riêng`, `Căn hộ chung cư`, `Văn phòng`), near-flat for `Đất`, and
slightly negative for the tiny `Khách sạn` group — versus the ≈ 0 pooled correlation in
10.2.1.
**Interpretation.** Area is a **conditional** driver: it predicts price *given* the
property type. Ignoring type produces Simpson's paradox. A per-type regression with the
fitted slopes is in 10.3.4.
**ML implication.** Keep `Area`, but the model must combine it with `Property Type` —
tree ensembles do this by splitting on type first; a linear model would need explicit
interaction terms.

In [ ]:
# ---- 10.2.4 · numeric correlation with log Price (Pearson) ----
num_corr_cols = ['price_log', 'area_log', 'Width', 'Length', 'Bedrooms',
                 'Bathrooms', 'Floors', 'Alley Width', 'Agent Listing Count']
cm = df[num_corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(cm, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(num_corr_cols))); ax.set_xticklabels(num_corr_cols, rotation=90)
ax.set_yticks(range(len(num_corr_cols))); ax.set_yticklabels(num_corr_cols)
for r in range(len(num_corr_cols)):
    for c in range(len(num_corr_cols)):
        ax.text(c, r, f'{cm.iat[r, c]:.2f}', ha='center', va='center', fontsize=8,
                color='white' if abs(cm.iat[r, c]) > 0.5 else 'black')
ax.set_title('10.2.4 · numeric correlation with log Price')
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

print('|r| with log Price:')
print(cm['price_log'].drop('price_log').abs().sort_values(ascending=False).round(3).to_string())

**Observation.** On the pooled data the strongest numeric correlation with `log Price` is
only `Bathrooms` ≈ 0.22; `area_log` ≈ 0.02. No pair of features has |r| ≈ 1.
**Interpretation.** The weak numbers are a pooling artefact (10.2.1), not evidence that
the features are useless. The absence of |r| ≈ 1 means no numeric column is redundant.
**ML implication.** Keep all numeric features; no need to drop any for collinearity. A
numeric-only view understates the dataset — see 10.2.5.

In [ ]:
# ---- 10.2.5 · categorical → log Price strength (correlation ratio η²) ----
cat_cols = ['Property Type', 'Province', 'Position', 'Direction', 'Road Type', 'Agent Role']
eta = pd.Series({c: correlation_ratio(df[c].fillna('Unknown'), df['price_log'])
                for c in cat_cols}).sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
eta.plot(kind='barh', ax=ax, color='#4C72B0')
for i, v in enumerate(eta.values):
    ax.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=9)
ax.set_xlim(0, max(0.3, eta.max() * 1.25))
ax.set_title('10.2.5 · categorical → log Price strength (η², variance explained)')
ax.set_xlabel('η²  (0 = no effect, 1 = fully explains price)')
plt.tight_layout(); plt.show()

print(eta.sort_values(ascending=False).round(3).to_string())

**Observation.** The correlation ratio η² (comparable to the numeric r²) shows
`Province` ≈ 0.24 and `Property Type` ≈ 0.12 — each explains **more** price variance than
any single numeric feature — while `Direction`, `Position`, `Road Type`, `Agent Role`
explain almost nothing (η² < 0.05).
**Interpretation.** Location and type are the real signal in this dataset; the numeric
Pearson heatmap alone is misleading.
**ML implication.** One-hot `Property Type` and `Province` / `district` (high η²);
`Direction` / `Position` / `Agent Role` (low η²) are kept for completeness but are the
first candidates to drop if `d` must shrink.

### 10.3 Supporting analyses for the modelling choices

Four short views, each justifying one later decision:

| # | View | Justifies |
|---|---|---|
| 10.3.1 | missing values per column | which columns to impute vs drop |
| 10.3.2 | missing values by property type | the `*_missing` indicator features (section 13) |
| 10.3.3 | skew before/after `log1p` + QQ-plot | the `log1p(Price)` target and the p99 caps |
| 10.3.4 | `log Price` vs `log Area` per property type | why the model needs area×type, not area alone |

In [ ]:
# ---- 10.3.1 · how much is missing, per column ----
miss_pct = (df.isna().mean() * 100)
miss_pct = miss_pct[miss_pct > 0].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 4.5))
miss_pct.plot(kind='barh', ax=ax, color='#C44E52')
ax.invert_yaxis()
ax.set_title('10.3.1 · missing values per column (%)')
ax.set_xlabel('% of rows missing')
for i, v in enumerate(miss_pct.values):
    ax.text(v + 1, i, f'{v:.0f}%', va='center', fontsize=8)
plt.tight_layout(); plt.show()

print(miss_pct.round(1).to_string())

**Observation.** `Bathrooms` ≈ 83%, `Floors` ≈ 77%, `Bedrooms` ≈ 70%,
`Alley Width` / `Direction` / `Road Type` ≈ 65–68%, `Latitude` / `Longitude` ≈ 73% of
rows are missing; `Area` is essentially complete after section 8.
**Interpretation.** More than half the candidate columns are majority-missing, so
row-dropping is not an option — it would delete most of the data.
**ML implication.** Numeric columns are median-imputed inside the pipeline (section 15);
`Latitude` / `Longitude` are too sparse to impute meaningfully and are dropped (province
and district cover location instead).

In [ ]:
# ---- 10.3.2 · is the missingness random? break it down by property type ----
flag_cols = ['Bedrooms', 'Bathrooms', 'Floors', 'Width', 'Length',
             'Alley Width', 'Direction', 'Road Type']
by_type = (df.groupby('Property Type')[flag_cols]
             .apply(lambda g: g.isna().mean() * 100).round(0))

fig, ax = plt.subplots(figsize=(10, 4.5))
im = ax.imshow(by_type.values, cmap='Reds', vmin=0, vmax=100, aspect='auto')
ax.set_xticks(range(len(flag_cols))); ax.set_xticklabels(flag_cols, rotation=45, ha='right')
ax.set_yticks(range(len(by_type.index))); ax.set_yticklabels(by_type.index)
for r in range(by_type.shape[0]):
    for cc in range(by_type.shape[1]):
        val = by_type.values[r, cc]
        ax.text(cc, r, f'{val:.0f}', ha='center', va='center', fontsize=8,
                color='white' if val > 55 else 'black')
ax.set_title('10.3.2 · % missing by Property Type')
plt.colorbar(im, ax=ax, fraction=0.046, label='% missing')
plt.tight_layout(); plt.show()

**Observation.** The missingness is clearly **not random**: for `Đất` (land)
`Bedrooms` is 99% missing and `Bathrooms` 100%; for `Căn hộ chung cư` (apartments)
`Floors` is 98% missing — a flat has no "number of floors"; `Nhà riêng` (houses) has
the lowest missing rates across the board.
**Interpretation.** Whether a field was filled in encodes the property type and how
carefully the listing was written — it is informative, not just absent.
**ML implication.** Median-imputing alone would throw that signal away, so section 13
adds `Bedrooms_missing` / `Bathrooms_missing` / `Floors_missing` binary indicator
columns alongside the imputed values.

In [ ]:
# ---- 10.3.3 · skew of the numeric columns, and what log1p does to it ----
from scipy import stats

skew_cols = ['Price', 'Area', 'Width', 'Length']
rows = [[c, stats.skew(df[c].dropna()), stats.skew(np.log1p(df[c].dropna()))] for c in skew_cols]
skew_tbl = pd.DataFrame(rows, columns=['column', 'skew_raw', 'skew_log1p']).round(2)
print(skew_tbl.to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
x = np.arange(len(skew_cols)); w = 0.38
ax[0].bar(x - w/2, skew_tbl['skew_raw'],   w, label='raw',   color='#C44E52')
ax[0].bar(x + w/2, skew_tbl['skew_log1p'], w, label='log1p', color='#55A868')
ax[0].set_xticks(x); ax[0].set_xticklabels(skew_cols)
ax[0].axhline(0, color='k', lw=0.8)
ax[0].axhline(1, color='grey', ls=':', lw=0.8); ax[0].axhline(-1, color='grey', ls=':', lw=0.8)
ax[0].set_ylim(-2, 6)                      # clip the ~380 raw Width/Length skew for readability
ax[0].set_ylabel('skewness'); ax[0].set_title('left · skew before vs after log1p'); ax[0].legend()

stats.probplot(df['price_log'], dist='norm', plot=ax[1])
ax[1].set_title('right · QQ-plot of log1p(Price) vs Normal')
plt.tight_layout(); plt.show()

**Observation.** Raw skew is `Price` 3.4, `Area` 5.1, `Width` / `Length` above 250 (a
few extreme but in-range values). After `log1p` every column falls to roughly ±1.3 and
`log1p(Price)` sits at ≈ 0 (left panel). The QQ-plot (right panel) shows `log1p(Price)` is close
to Normal through the body, bending only in the far tails.
**Interpretation.** On the raw scale a handful of large values would dominate any
squared-error loss and distort linear-model coefficients.
**ML implication.** The model is trained on `log1p(Price)` and scored after `expm1`
(sections 12, 16); `Area` / `Width` / `Length` / `Alley Width` are additionally capped at
the 99th percentile in section 13.

In [ ]:
# ---- 10.3.4 · does Area predict Price WITHIN each property type? ----
types = [t for t in df['Property Type'].value_counts().index
         if (df['Property Type'] == t).sum() >= 200]

n = len(types); ncol = 3; nrow = int(np.ceil(n / ncol))
fig, ax = plt.subplots(nrow, ncol, figsize=(16, 3.2 * nrow), squeeze=False)
slopes = {}
for k, t in enumerate(types):
    a = ax[k // ncol][k % ncol]
    g = df[df['Property Type'] == t]
    gs = g.sample(min(2500, len(g)), random_state=RANDOM_SEED)
    a.scatter(gs['area_log'], gs['price_log'], s=6, alpha=0.25)
    b, a0, r, _, _ = stats.linregress(g['area_log'], g['price_log'])
    slopes[t] = (b, r, len(g))
    xs = np.linspace(g['area_log'].min(), g['area_log'].quantile(0.99), 50)
    a.plot(xs, a0 + b * xs, 'r-', lw=1.5)
    a.set_title(f'{t}  (n={len(g):,})  slope={b:.2f}, r={r:.2f}', fontsize=9)
    a.set_xlabel('log1p(Area)'); a.set_ylabel('log1p(Price)')
for k in range(n, nrow * ncol):
    ax[k // ncol][k % ncol].axis('off')
plt.suptitle('10.3.4 · log Price ~ log Area, fitted separately per property type', y=1.02)
plt.tight_layout(); plt.show()

print('slope of log Price ~ log Area, by Property Type:')
for t, (b, r, m) in sorted(slopes.items(), key=lambda kv: -kv[1][0]):
    print(f'  {t:18s} n={m:6d}  slope={b:5.2f}  r={r:5.2f}')

**Observation.** Fitted per type, the `log Area` → `log Price` slope is clearly
positive for `Văn phòng` (≈ 0.99), `Căn hộ chung cư` (≈ 0.47), `Nhà riêng` (≈ 0.36) and
`Kho, nhà xưởng` (≈ 0.34); weak for `Đất` (≈ 0.14) and `Nhà trọ` (≈ 0.14); and roughly
zero for the small `Khách sạn` group. The pooled slope (10.2.1 / 10.2.4) was near zero.
**Interpretation.** This is **Simpson's paradox**: area does drive price, but only
*conditional* on property type — the land vs apartment mix cancels it out in aggregate.
**ML implication.** Additive linear models (which see only `Area`) will underperform;
tree ensembles, which split on `Property Type` first and then on `Area`, represent this
interaction natively. This is exactly the section 18 ranking (Random Forest / XGBoost
ahead of Linear / Ridge).

## 11. Feature types

In [ ]:
feature_types = {
    # --- numerical ---
    'Area': 'numerical', 'Width': 'numerical', 'Length': 'numerical',
    'Bedrooms': 'numerical (count)', 'Bathrooms': 'numerical (count)',
    'Floors': 'numerical (count)', 'Alley Width': 'numerical',
    'Agent Listing Count': 'numerical (count)',
    # --- categorical (one-hot) ---
    'Property Type': 'categorical', 'Position': 'categorical',
    'Direction': 'categorical', 'Road Type': 'categorical',
    'Province': 'categorical (63 levels)', 'Agent Role': 'categorical',
    # --- engineered from text (section 13) ---
    'Location': 'text -> parsed into the district feature, then dropped',
    # --- target ---
    'Price': 'target (continuous, million VND)',
    # --- dropped ---
    'Title': 'DROP — free text, leaks Price',
    'Description': 'DROP — free text, not used',
    'Listing ID': 'DROP — identifier (used only for dedupe)',
    'VIP Account': 'DROP — constant',
    'Avatar': 'DROP — not informative',
    'Agent Name': 'DROP — 28k unique, high-cardinality noise',
    'Property Type Slug': 'DROP — duplicate of Property Type',
    'Last Updated': 'DROP — free-text recency string',
    'Scraped At': 'DROP — scrape timestamp',
    'Last Updated Date': 'DROP — timestamp',
    'Latitude': 'DROP — 68% missing', 'Longitude': 'DROP — 68% missing',
}
pd.Series(feature_types, name='role')

**Result.** 8 numerical inputs, 6 one-hot categoricals, 1 text column parsed into the
`district` categorical (ward dropped as noise), 1 continuous target, 13 columns dropped
(identifiers, constants, free text, duplicates, sparse coords). Nothing dropped for
redundancy except `Property Type Slug` (identical information to `Property Type`).

## 12. Data representation

```
CSV  ->  DataFrame  ->  clean feature frame  ->  ColumnTransformer (impute + scale + one-hot)  ->  X  ->  model
```

The cell below prints one raw listing and the feature vector it becomes, together with
the DataFrame shape, the final `X` shape and its dtype.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# minimal engineered frame just for this demo (full version in section 13)
NUM_DEMO = ['Area', 'Width', 'Length', 'Bedrooms', 'Bathrooms', 'Floors', 'Alley Width']
CAT_DEMO = ['Property Type', 'Position', 'Direction', 'Road Type', 'Province', 'Agent Role']

demo_df = df[NUM_DEMO + CAT_DEMO].copy()
for c in CAT_DEMO:
    demo_df[c] = demo_df[c].fillna('Unknown')
y_demo = np.log1p(df['Price'].values)   # model target = log1p(Price)

demo_prep = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), NUM_DEMO),
    ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=20), CAT_DEMO),
])
X_demo = demo_prep.fit_transform(demo_df)

print('one RAW listing (selected columns):')
print(df.iloc[0][['Title', 'Price', 'Area', 'Property Type', 'Location']].to_dict())
print('\nsame listing as a FEATURE VECTOR x (first 15 dims of the imputed+scaled+one-hot row):')
row0 = X_demo[0].toarray().ravel() if hasattr(X_demo, 'toarray') else X_demo[0]
print(np.round(row0[:15], 3))
print('\nDataFrame shape at this point :', df.shape, ' (after sections 7-8 cleaning)')
print('feature matrix X shape   :', X_demo.shape, '  # X in R^{N x d}')
print('target vector y shape    :', y_demo.shape, '  # y in R^N  (log1p million VND)')
print('dtype of X               :', X_demo.dtype)
print('one API request is       :  R^{1 x d}   (a single listing)')

**Model input.** `X` is the array passed to `model.fit(X, y)` /
`model.predict(X)`. Shape `(N, d)`: `N` listings in the batch, `d` features **after**
median-imputing the numeric columns, standardising them, and one-hot encoding the six
categoricals (rare levels folded into `infrequent` by `min_frequency`). It differs from
the raw CSV: the target and all dropped columns are gone, missing values are filled,
categoricals became 0/1 indicator columns, numerics are mean-0/std-1. `y` is
`log1p(Price)` — predictions are converted back with `expm1`. `d` is finalised in
section 13. In the verified run `d = 404` (11 numeric + 393 one-hot indicator columns).

## 13. Feature engineering

In [ ]:
def parse_district(loc):
    """'Đường X,Phường Y,Quận Z,Tỉnh(Mới)' -> district.
    Prefer a Quận/Huyện/Thị xã/Thành phố token; fall back to the ward
    (Phường/Xã/Thị trấn/Đặc khu); 'Unknown' if neither is present.
    Ward itself is NOT used as a feature: ~4,600 levels is mostly noise."""
    if not isinstance(loc, str):
        return 'Unknown'
    parts = [p.strip() for p in loc.replace('(Mới)', '').split(',') if p.strip()]
    dist = next((p for p in parts if p.startswith(('Quận', 'Huyện', 'Thị xã', 'Thành phố', 'TP.', 'TP '))), None)
    if dist:
        return dist
    return next((p for p in parts if p.startswith(('Phường', 'Xã', 'Thị trấn', 'Đặc khu'))), 'Unknown')


def engineer(frame):
    out = frame.copy()
    # 1. parsed location (district level only)
    out['district'] = out['Location'].apply(parse_district)
    # 2. missing-indicator flags for the >50%-missing counts
    for c in ['Bedrooms', 'Bathrooms', 'Floors']:
        out[c + '_missing'] = out[c].isna().astype(int)
    # 3. cap extreme size (99th pct) so distance/linear models aren't dominated
    for c in ['Area', 'Width', 'Length', 'Alley Width']:
        cap = out[c].quantile(0.99)
        out[c] = out[c].clip(upper=cap)
    # 4. fill categorical NaN with an explicit level
    for c in ['Position', 'Direction', 'Road Type']:
        out[c] = out[c].fillna('Unknown')
    return out


df_eng = engineer(df)

NUM_FEATURES = ['Area', 'Width', 'Length', 'Bedrooms', 'Bathrooms', 'Floors',
                'Alley Width', 'Agent Listing Count',
                'Bedrooms_missing', 'Bathrooms_missing', 'Floors_missing']
CAT_FEATURES = ['Property Type', 'Position', 'Direction', 'Road Type',
                'Province', 'Agent Role', 'district']
FEATURES = NUM_FEATURES + CAT_FEATURES

print('final raw feature count :', len(FEATURES))
print('numeric  :', NUM_FEATURES)
print('categoric:', CAT_FEATURES)
print('district levels        :', df_eng['district'].nunique(),
      '(rare ones folded by min_frequency in section 15)')
df_eng[FEATURES].head(3)

**Justification.**

| Engineered feature | Why |
|---|---|
| `district` (from `Location`) | location is the second-biggest price driver; the raw string is unusable. District is the useful admin level (~2,000 raw levels); `ward` (~4,600) is dropped as noise. One-hot with `min_frequency=50` folds the long tail so `d` stays bounded. |
| `Bedrooms_missing` / `Bathrooms_missing` / `Floors_missing` | these columns are >70% missing and absence correlates with property type (land listings). The flag lets the model use "was this recorded?" instead of trusting the imputed median. |
| 99th-percentile cap on `Area` / `Width` / `Length` / `Alley Width` | a handful of 5,000 m² lots give linear / KNN models huge leverage; capping keeps every row while bounding influence. |

**Encoding choice.** All six original categoricals + the two parsed ones use **one-hot**
(`OneHotEncoder(handle_unknown='ignore', min_frequency=...)`). One-hot, not ordinal,
because none of these have a natural order. `Province` (63) and `district` (hundreds)
rely on `min_frequency` to cap the column count. The fitted pipeline (section 15) produces **`d = 404`**.

## 14. Train / validation / test split

In [ ]:
from sklearn.model_selection import train_test_split

X_all = df_eng[FEATURES].copy()
y_all = np.log1p(df_eng['Price'].values)          # model target
y_all_vnd = df_eng['Price'].values                # kept for reporting on the real scale

X_train, X_tmp, y_train, y_tmp, _, y_tmp_vnd = train_test_split(
    X_all, y_all, y_all_vnd, test_size=0.30, random_state=RANDOM_SEED)
X_val, X_test, y_val, y_test, y_val_vnd, y_test_vnd = train_test_split(
    X_tmp, y_tmp, y_tmp_vnd, test_size=0.50, random_state=RANDOM_SEED)

print(f'train: {X_train.shape}   val: {X_val.shape}   test: {X_test.shape}   (~70/15/15)')

**Why test data must not influence training or preprocessing fitting (data leakage).**
If the imputer's medians, the scaler's mean/std, or the one-hot vocabulary were computed
over rows that later appear in validation/test, the model would indirectly "know" those
rows and the reported error would be optimistically low — it would not hold on genuinely
new listings. So the split happens **before** any `.fit()`, and every preprocessing step
is fitted on `X_train` only (section 15). Regression needs no stratification, but the
`Listing ID` dedupe in section 7 already prevents the *same* property landing on both
sides. `random_state` fixed for reproducibility.

## 15. Preprocessing pipeline

In [ ]:
numeric_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
])
categorical_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=50, sparse_output=True)),
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipe, NUM_FEATURES),
    ('cat', categorical_pipe, CAT_FEATURES),
])

preprocessor.fit(X_train)                    # <-- fitted on TRAIN ONLY
X_train_pp = preprocessor.transform(X_train)
D_FEATURES = X_train_pp.shape[1]
print('preprocessed train shape:', X_train_pp.shape, '  # d =', D_FEATURES)
print('(numeric', len(NUM_FEATURES), '+ one-hot', D_FEATURES - len(NUM_FEATURES),
      'indicator columns, rare levels folded into an <infrequent> column)')

**Steps and purpose.** (1) numeric: median-impute the missing sizes/counts, then
`StandardScaler` so linear regression, SVR and KNN treat every feature on a comparable
scale; (2) categorical: fill `'Unknown'`, then one-hot with `min_frequency=50` so rare
districts / provinces collapse into an `infrequent` column instead of exploding `d`.
This one fitted `preprocessor` is bundled with the chosen model in section 22 and loaded
**unchanged** by `../api/` — deployment never re-fits it.

## 16. Baseline model

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

LOG_CLIP = np.log1p(1_000_000)   # cap predictions at 1,000,000 million VND before expm1

def _to_vnd(y_log):
    """Invert the log1p target, guarding against overflow from wild linear extrapolation."""
    return np.expm1(np.clip(y_log, None, LOG_CLIP))

def regression_scores(y_true_log, y_pred_log):
    """Scores reported on the real VND scale (million VND)."""
    yt, yp = _to_vnd(y_true_log), _to_vnd(y_pred_log)
    mae = mean_absolute_error(yt, yp)
    mse = mean_squared_error(yt, yp)
    rmse = np.sqrt(mse)
    r2 = r2_score(yt, yp)
    mape = np.mean(np.abs((yt - yp) / np.clip(yt, 1e-6, None))) * 100
    return {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2, 'MAPE_%': mape}

base = Pipeline([('prep', preprocessor), ('reg', DummyRegressor(strategy='median'))])
base.fit(X_train, y_train)
base_scores = regression_scores(y_val, base.predict(X_val))
print('baseline (predict median log-price):')
for k, v in base_scores.items():
    print(f'  {k:7s}: {v:,.3f}')

**Reference score.** Always predicting the median log-price gives MAE ≈ 14,400 million
VND and R² ≈ −0.13 on the validation set. Every trained model below must beat this; a
model that cannot is no better than a constant guess.

## 17. Model training

Five regression models are compared: Linear Regression, Ridge, Decision Tree, Random
Forest and Gradient Boosting (XGBoost). All share the identical `preprocessor` and the
same `X_train` / `y_train` (log-transformed target).

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
try:
    from xgboost import XGBRegressor
    _HAS_XGB = True
except ImportError:
    from sklearn.ensemble import GradientBoostingRegressor
    _HAS_XGB = False

# Train all models on the SAME sample so the comparison is fair and the notebook
# runs in a few minutes. Raise / remove TRAIN_SAMPLE for a final run.
TRAIN_SAMPLE = 50_000
if len(X_train) > TRAIN_SAMPLE:
    samp_idx = np.random.RandomState(RANDOM_SEED).choice(len(X_train), TRAIN_SAMPLE, replace=False)
    X_tr_s, y_tr_s = X_train.iloc[samp_idx], y_train[samp_idx]
else:
    X_tr_s, y_tr_s = X_train, y_train
print(f'training sample: {X_tr_s.shape[0]:,} rows')

models = {
    'LinearRegression': LinearRegression(),
    'Ridge'           : Ridge(alpha=10.0, random_state=RANDOM_SEED),
    'DecisionTree'    : DecisionTreeRegressor(max_depth=12, min_samples_leaf=25, random_state=RANDOM_SEED),
    'RandomForest'    : RandomForestRegressor(n_estimators=120, max_depth=18, min_samples_leaf=10,
                                              n_jobs=-1, random_state=RANDOM_SEED),
}
if _HAS_XGB:
    models['XGBoost'] = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                     subsample=0.8, colsample_bytree=0.8, tree_method='hist',
                                     objective='reg:squarederror', random_state=RANDOM_SEED, n_jobs=-1)
else:
    models['GradientBoosting'] = GradientBoostingRegressor(random_state=RANDOM_SEED)

fitted, timings = {}, {}
for name, reg in models.items():
    pipe = Pipeline([('prep', preprocessor), ('reg', reg)])
    t0 = time.time(); pipe.fit(X_tr_s, y_tr_s); dt = time.time() - t0
    fitted[name] = pipe; timings[name] = dt
    print(f'{name:18s} trained in {dt:6.2f}s')

**Note.** Tree models (`DecisionTree`, `RandomForest`, `XGBoost`) don't need the scaling,
but running them through the same `preprocessor` keeps one code path and one persisted
object. `max_depth` / `min_samples_leaf` are set conservatively to limit overfitting; a
hyper-parameter search could be added but was not required for the comparison below.

## 18. Model comparison

In [ ]:
rows = []
for name, pipe in fitted.items():
    s = regression_scores(y_val, pipe.predict(X_val))
    s['model'] = name; s['train_s'] = round(timings[name], 2)
    rows.append(s)
cmp = pd.DataFrame(rows).set_index('model')[['MAE', 'MSE', 'RMSE', 'R2', 'MAPE_%', 'train_s']]
cmp = cmp.sort_values('RMSE')
cmp.round(3)

**Which model leads.** Random Forest gives the lowest validation RMSE and the highest
R², with XGBoost close behind. Linear Regression and Ridge trail because they cannot
capture the area×location×type interactions; the single Decision Tree sits in between.
Random Forest is carried to the held-out test set.

## 19. Evaluation

*(on the held-out test set — the split fixed in section 14, never touched during training or model comparison.)*

In [ ]:
BEST = cmp.index[0]          # model with the lowest validation RMSE
print('selected model:', BEST)

final_pipe = fitted[BEST]
test_scores = regression_scores(y_test, final_pipe.predict(X_test))
for k, v in test_scores.items(): print(f'  {k:7s}: {v:,.3f}')

# predicted vs actual (VND scale)
yp = np.expm1(final_pipe.predict(X_test)); yt = y_test_vnd
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].scatter(yt, yp, s=6, alpha=0.3); lim = [0, np.percentile(yt, 99)]
ax[0].plot(lim, lim, 'r--'); ax[0].set_xlim(lim); ax[0].set_ylim(lim)
ax[0].set_xlabel('actual Price (million VND)'); ax[0].set_ylabel('predicted'); ax[0].set_title('Predicted vs Actual')
resid = yp - yt
ax[1].scatter(yt, resid, s=6, alpha=0.3); ax[1].axhline(0, color='r', ls='--')
ax[1].set_xlim(lim); ax[1].set_xlabel('actual Price'); ax[1].set_ylabel('residual (pred - actual)')
ax[1].set_title('Residuals vs Actual')
plt.tight_layout(); plt.show()

**Metrics interpreted (what each means for this application).**

- **MAE** — the average absolute miss in **million VND**. "On a typical listing the
  estimate is off by ~X tỷ." The number a user feels.
- **MSE / RMSE** — squared-error based, so large misses (mispricing a villa) are punished
  far more than small ones. RMSE is back on the VND scale and is the headline error.
- **R²** — share of price variance the model explains vs. always guessing the mean
  (1 = perfect, 0 = no better than the mean). In the verified run the test R² is ≈ 0.19.
- **MAPE** — average *percentage* error; comparable across cheap and expensive listings.

There is no confusion matrix — this is a regression task. The predicted-vs-actual plot
should follow the diagonal; the residual plot shows a widening spread at higher prices
(heteroscedasticity), i.e. the model's absolute error grows for more expensive listings.

## 20. Error analysis

In [ ]:
err = X_test.copy()
err['actual'] = y_test_vnd
err['pred'] = np.expm1(final_pipe.predict(X_test))
err['abs_err'] = (err['pred'] - err['actual']).abs()
err['pct_err'] = err['abs_err'] / err['actual'] * 100

print('worst 10 absolute errors:')
display(err.sort_values('abs_err', ascending=False).head(10)[
    ['Property Type', 'Province', 'district', 'Area', 'actual', 'pred', 'abs_err', 'pct_err']])

print('\nmean |%err| by property type:')
display(err.groupby('Property Type')['pct_err'].mean().sort_values(ascending=False).round(1))

print('\nmean |%err| by area band:')
err['area_band'] = pd.cut(err['Area'], [0, 50, 100, 200, 500, 10000])
display(err.groupby('area_band')['pct_err'].mean().round(1))

**What the model struggles with.** The largest errors fall on (i) very large or very
cheap listings, where training data is thin at the extremes; (ii) sparse property types
such as `Kho, nhà xưởng` (mean absolute percentage error well above the others); and
(iii) districts folded into the `infrequent` one-hot bucket, which lose their local
price level. Likely causes: representation gap
(no true street-level location, no year-built / legal-status), skewed target, sparse
segments. Possible fixes: target-encode `district`, add missing attributes, collect more
rows for rare types, or a separate model per property type.

## 21. Model selection

**Deployed model: Random Forest** — the lowest test RMSE and highest R² of the five.

| Criterion | Random Forest | Note |
|---|---|---|
| Predictive performance | lowest test RMSE, highest R² (≈ 0.19), MAE ≈ 10,600 million VND | primary reason |
| Interpretability | feature importances available | can report which features drove an estimate |
| Computational cost | trains in ~15 s on the 50,000-row sample; predicts in a few ms per listing | fine for a request/response API |
| Robustness | bagging absorbs the price skew and the mild outliers left after section 8 | |
| Deployment constraints | ~45 MB when pickled, CPU-only | container-friendly |

Linear Regression and Ridge are kept as a documented fallback: much smaller and fully
interpretable, but clearly higher error.

## 22. Model persistence

In [ ]:
import joblib, os, json

os.makedirs('../model', exist_ok=True)

# refit the chosen pipeline on train + val so deployment uses all non-test data
deploy_pipe = Pipeline([('prep', preprocessor), ('reg', models[BEST])])
# refit on the training sample + validation set (consistent with section 17)
X_fit = pd.concat([X_tr_s, X_val]); y_fit = np.concatenate([y_tr_s, y_val])
deploy_pipe.fit(X_fit, y_fit)

joblib.dump(deploy_pipe, '../model/model_pipeline.joblib')
joblib.dump(FEATURES,    '../model/feature_names.joblib')

schema = {
    'target': 'Price (million VND); model trained on log1p(Price), invert with expm1',
    'task': 'regression',
    'chosen_model': BEST,
    'random_seed': RANDOM_SEED,
    'sklearn_version': sklearn.__version__,
    'price_unit': 'million VND',
    'numeric_features': NUM_FEATURES,
    'categorical_features': CAT_FEATURES,
    'model_features_order': FEATURES,
    'engineered': ['district (parsed from Location: Quận/Huyện/Thị xã/TP token, else ward, else Unknown)',
                   'Bedrooms_missing', 'Bathrooms_missing', 'Floors_missing',
                   '99th-pct cap on Area/Width/Length/Alley Width'],
    'dropped_columns': ['Title (leaks Price)', 'Description', 'Listing ID', 'VIP Account',
                        'Avatar', 'Agent Name', 'Property Type Slug', 'Last Updated',
                        'Scraped At', 'Last Updated Date', 'Latitude', 'Longitude', 'Location'],
}
with open('../model/input_schema.json', 'w') as f:
    json.dump(schema, f, ensure_ascii=False, indent=2)

print('saved ../model/model_pipeline.joblib   (preprocessing + regressor in one object)')
print('saved ../model/feature_names.joblib    (expected raw input columns, ordered)')
print('saved ../model/input_schema.json       (input contract for ../api/)')

**Files produced.** `model/model_pipeline.joblib` — one object holding the fitted
imputers + scaler + one-hot encoder + the chosen regressor. `model/feature_names.joblib`
— the ordered raw column list `../api/` validates against. `model/input_schema.json` —
the human-readable contract. Nothing else is needed at inference; the API loads exactly
these and re-fits nothing.

## 23. Inference test

In [ ]:
loaded_pipe     = joblib.load('../model/model_pipeline.joblib')
loaded_features = joblib.load('../model/feature_names.joblib')

raw_listing = {          # exactly what the API / mobile client sends (raw, unprocessed)
    'Area': 78.7, 'Width': 4.0, 'Length': np.nan,
    'Bedrooms': 3, 'Bathrooms': 2, 'Floors': 2, 'Alley Width': np.nan,
    'Agent Listing Count': 2,
    'Property Type': 'Nhà riêng', 'Position': 'Đường chính', 'Direction': 'Nam',
    'Road Type': 'Đường nhựa', 'Province': 'an-giang', 'Agent Role': 'Chính chủ',
    'district': 'Rạch Giá',
}
# the engineered missing-flags the client can't know — derive them the same way the notebook did
for c in ['Bedrooms', 'Bathrooms', 'Floors']:
    raw_listing[c + '_missing'] = int(pd.isna(raw_listing.get(c)))

row = pd.DataFrame([raw_listing])[loaded_features]     # right columns, right order
pred_log = float(loaded_pipe.predict(row)[0])
pred_price = float(np.expm1(pred_log))                 # back to million VND

result = {
    'predicted_price': round(pred_price, 2),
    'price_per_m2': round(pred_price * 1000 / raw_listing['Area'], 2),
    'currency': 'million VND',
    'model': BEST,
}
print(result)

**Contract confirmed.** The reloaded pipeline (fresh from disk) turns a **raw dict**
into `{ "predicted_price": <million VND>, "price_per_m2": ..., "model": ... }` via
`raw input -> validate columns -> the same fitted preprocessing -> regressor -> expm1`.
**No new scaler / imputer / encoder was fitted here.** This is exactly what `../api/`
does — its `POST /predict` endpoint wraps this same call.